In [23]:
%load_ext autoreload
%autoreload 2

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Set default plotly template
pio.templates.default = "plotly_white"

In [2]:
df = pd.read_csv("speeches_with_nli_and_party.csv")

In [ ]:
# Swiss party colors for consistent, intuitive visualization
PARTY_COLORS = {
    "SP": "#E30613",  # Red
    "Grüne": "#84B414",  # Green
    "GLP": "#CEE00F",  # Light Green/Yellow
    "Mitte": "#D7A200",  # Orange/Yellow
    "FDP-Liberale": "#0076BD",  # Blue
    "SVP": "#418F4E",  # Dark Green
}

# Ensure data only has the major parties for clean plotting
major_parties = ["SP", "Grüne", "GLP", "Mitte", "FDP-Liberale", "SVP"]
df_major = df[df["party"].isin(major_parties)].copy()
spectrum_order = ["SP", "Grüne", "GLP", "Mitte", "FDP-Liberale", "SVP"]
df_major["party"] = pd.Categorical(
    df_major["party"], categories=spectrum_order, ordered=True
)

In [ ]:
# Global plotly layout
pub_layout = dict(
    font=dict(family="Times New Roman, serif", size=14),
    margin=dict(l=50, r=50, t=80, b=50),
    hovermode="closest",
)

In [46]:
fig = px.histogram(
    df,
    x="Stance_Intensity",
    nbins=100,
    color_discrete_sequence=["#2c7bb6"],
    title=r"Distribution of Legislative Stance Intensity $(P_{pro} - P_{con})$",
    labels={"Stance_Intensity": "Stance Score (Opposition ← Neutral → Advocacy)"},
)

fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.6)
fig.add_vline(
    x=df["Stance_Intensity"].mean(),
    line_color="orange",
    annotation_text=f"Mean: {df['Stance_Intensity'].mean():.2f}",
)
fig.add_vline(
    x=df["Stance_Intensity"].median(),
    line_color="purple",
    annotation_text=f"Median: {df['Stance_Intensity'].median():.2f}",
)

fig.update_layout(
    **pub_layout,
    showlegend=False,
    xaxis_title="Stance Score (Opposition ← Neutral → Advocacy)",
    yaxis_title="Number of Paragraphs",
)

# Add region labels
fig.add_annotation(
    x=-0.8,
    y=0.9,
    yref="paper",
    text="Strong Opposition",
    font=dict(color="darkred", size=14, weight="bold"),
    showarrow=False,
)
fig.add_annotation(
    x=0.8,
    y=0.9,
    yref="paper",
    text="Strong Advocacy",
    font=dict(color="darkgreen", size=14, weight="bold"),
    showarrow=False,
)
fig.add_annotation(
    x=0.05,
    y=0.85,
    yref="paper",
    text="Neutral/Procedural",
    font=dict(color="gray", size=12),
    showarrow=False,
)

fig.show()

In [5]:
df["abs_intensity"] = df["Stance_Intensity"].abs()

In [6]:
# Group by Party and Topic
party_advocacy = (
    df.groupby(["party", "Representation_with_Direction"])
    .agg(
        avg_stance=("Stance_Intensity", "mean"),
        avg_priority=("abs_intensity", "mean"),
        speech_count=("TEXT_SPAN", "count"),
    )
    .reset_index()
)

In [7]:
party_advocacy["advocacy_index"] = (
    party_advocacy["speech_count"] * party_advocacy["avg_stance"]
)

In [8]:
party_advocacy[party_advocacy["party"] == "SVP"].sort_values(
    by="advocacy_index", ascending=False
).head(10)

,party,Representation_with_Direction,avg_stance,avg_priority,speech_count,advocacy_index
1152,SVP,"Expanding national security, armed forces and ...",0.224976,0.521902,2065,464.575007
1225,SVP,Strengthening Swiss engagement in internationa...,0.283031,0.421160,1133,320.674076
1233,SVP,Strengthening data protection and privacy rights,0.216764,0.401762,1031,223.484069
1211,SVP,Reforming social security to ensure long-term ...,0.227035,0.331391,825,187.303980
1153,SVP,Expanding prevention measures against terrorism,0.428194,0.511396,313,134.024876
1204,SVP,Reducing health insurance costs,0.087704,0.276837,1176,103.139716
1166,SVP,Improving international security and human rights,0.260321,0.297120,396,103.086968
1140,SVP,Expanding Swiss research and innovation,0.251035,0.319415,401,100.664843
1164,SVP,Improving Swiss-EU relations,0.268960,0.388363,332,89.294879
1141,SVP,Expanding Switzerland's bilateral trade agreem...,0.286522,0.422076,302,86.529765


In [9]:
top_topics = df["Representation_with_Direction"].value_counts().index
pivot_df = party_advocacy[
    party_advocacy["Representation_with_Direction"].isin(top_topics)
].pivot(index="party", columns="Representation_with_Direction", values="avg_stance")

In [10]:
# Comprehensive Stance Heatmap (Plotly)
fig = px.imshow(
    pivot_df.T,
    labels=dict(x="Party", y="Topic", color="Mean Stance Intensity"),
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Mean Stance Intensity by Party and Topic",
)

fig.update_layout(
    height=1200,  # Large height for many topics
    width=1000,
    xaxis_tickangle=-90,
    **pub_layout,
)

fig.show()
# pio.write_image(fig, "party_stance_heatmap.pdf")

In [11]:
# Ideological Clustering of Major Swiss Parties (Plotly Heatmap)
major_parties = ["SP", "Grüne", "GLP", "Mitte", "FDP-Liberale", "SVP"]
pivot_major = pivot_df.loc[major_parties].fillna(0)

fig = px.imshow(
    pivot_major.T,
    labels=dict(x="Party", y="Topic", color="Mean Stance Intensity"),
    x=major_parties,
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Ideological Positioning of Major Swiss Parties",
)

fig.update_layout(**pub_layout, height=800, width=600, xaxis_tickangle=-45)

fig.show()

In [12]:
# Figure 1: Distribution of Inferred Legislative Stances by Party (Plotly)
fig = px.violin(
    df_major,
    x="party",
    y="Stance_Intensity",
    color="party",
    color_discrete_map=PARTY_COLORS,
    box=True,
    points=False,
    category_orders={"party": spectrum_order},
    labels={"Stance_Intensity": "NLI Stance Intensity", "party": "Party"},
    title="Distribution of Inferred Legislative Stances by Party",
)

fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

fig.update_layout(
    **pub_layout,
    showlegend=False,
    yaxis_title="NLI-Predicted Stance Intensity ($P_{pro} - P_{con}$)",
    xaxis_title="",
)

# Add annotations for polarity
fig.add_annotation(
    x=2.5,
    y=1.0,
    text="Advocacy Framing →",
    showarrow=False,
    font=dict(color="darkgreen", size=12),
    yref="paper",
)
fig.add_annotation(
    x=2.5,
    y=0.0,
    text="← Oppositional Framing",
    showarrow=False,
    font=dict(color="darkred", size=12),
    yref="paper",
)

fig.show()
# pio.write_image(fig, "stance_by_party_violin.pdf")

### The Ideological Heatmap

In [14]:
# Figure 2: Party Stances on Top 20 Most Salient Topics (Plotly Heatmap)
# Select top 20 topics by total speech volume for a cleaner plot
top_20_topics = (
    df_major["Representation_with_Direction"].value_counts().nlargest(20).index
)

heatmap_data = (
    df_major[df_major["Representation_with_Direction"].isin(top_20_topics)]
    .groupby(["Representation_with_Direction", "party"])["Stance_Intensity"]
    .mean()
    .unstack(level="party")
)

heatmap_data = heatmap_data[spectrum_order]  # left-to-right sorting

fig = px.imshow(
    heatmap_data,
    labels=dict(x="Party", y="Topic", color="Mean Stance Intensity"),
    x=spectrum_order,
    y=heatmap_data.index,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Party Stances on Top 20 Most Salient Topics",
)

fig.update_traces(text=heatmap_data.values, texttemplate="%{text:.2f}")

fig.update_layout(**pub_layout, xaxis_tickangle=-45, height=1200, width=1200)

fig.show()
# pio.write_image(fig, "party_stance_top20_heatmap.pdf")

### Topic Polarization

In [33]:
TOP_N = 10
# Calculate std dev of party means for all topics with > 100 overall items
topic_counts = df_major["Representation_with_Direction"].value_counts()
valid_topics = topic_counts[topic_counts > 100].index

party_means = (
    df_major[df_major["Representation_with_Direction"].isin(valid_topics)]
    .groupby(["Representation_with_Direction", "party"])["Stance_Intensity"]
    .mean()
    .unstack()
)

topic_polarization = party_means.std(axis=1).sort_values(ascending=False)
top_polarized = topic_polarization.head(TOP_N).index[::-1]  # Reverse for bottom-up plot

fig = go.Figure()

for idx, topic in enumerate(top_polarized):
    topic_data = party_means.loc[topic]

    # Add range line
    fig.add_trace(
        go.Scatter(
            x=[topic_data.min(), topic_data.max()],
            y=[topic, topic],
            mode="lines",
            line=dict(color="lightgray", width=2),
            showlegend=False,
            hoverinfo="skip",
        )
    )

    # Add party points
    for party in spectrum_order:
        val = topic_data[party]
        if not pd.isna(val):
            fig.add_trace(
                go.Scatter(
                    x=[val],
                    y=[topic],
                    mode="markers",
                    marker=dict(
                        color=PARTY_COLORS[party],
                        size=12,
                        line=dict(width=1, color="white"),
                    ),
                    name=party,
                    legendgroup=party,
                    showlegend=(idx == 0),
                    hovertemplate=f"Party: {party}<br>Stance: {val:.2f}<extra></extra>",
                )
            )

fig.add_vline(x=0, line_dash="dash", line_color="black", opacity=0.5)

fig.update_layout(
    **pub_layout,
    height=700,
    width=1200,
    xaxis_title="Mean Party Stance Intensity",
    yaxis_title="",
    title=f"Top {TOP_N} Most Polarized Legislative Issues",
    legend_title="Party",
    legend=dict(y=0.5, x=1.05),
)

fig.show()
# pio.write_image(fig, "topic_polarization_forestplot.pdf")